In [ ]:
# Install required dependencies for Google Colab
!pip install git+https://github.com/elkins-lab/synth-saxs.git biotite matplotlib py3Dmol ipywidgets

# Interactive SAXS Simulator Sandbox

Welcome to the `synth-saxs` interactive sandbox! In this notebook, you can explore how different environmental parameters—like the density of the solvent and the hydration shell—affect the resulting Small-Angle X-ray Scattering (SAXS) profile of a protein.

This intuition is critical for structural biologists:
1. **Solvent Density**: The baseline scattering of the buffer. SAXS measures the *contrast* between the protein and this solvent.
2. **Hydration Shell Density**: Water molecules immediately surrounding the protein are often tightly packed and denser than bulk water. This uniquely affects high-$q$ scattering.

**Instructions:**
- Run the cells below to load the structure.
- Use the interactive sliders to adjust the parameters in real-time!


In [ ]:
import biotite.database.rcsb as rcsb
import biotite.structure.io as strucio
import matplotlib.pyplot as plt
import py3Dmol
from ipywidgets import FloatSlider, interact

from synth_saxs.engine import calculate_saxs_profile, preprocess_structure

# Ensure inline plotting
%matplotlib inline

## 1. Load the Structure (Lysozyme)
We will use Hen Egg-White Lysozyme (PDB: 1AKI), a classic, small, and highly stable folded protein often used as a SAXS standard.

In [ ]:
print("Downloading 1AKI from PDB...")
pdb_file = rcsb.fetch("1AKI", "pdb", target_path="1AKI.pdb")
structure = strucio.load_structure(pdb_file)

# Coarse-grain to C-alpha atoms for extremely fast real-time interaction
structure = preprocess_structure(structure)
# Coarse-grain to CA atoms for speed
if "atom_name" in structure.get_annotation_categories():
    structure = structure[structure.atom_name == "CA"]
print(f"Loaded {len(structure)} atoms.")

## 2. 3D Visualization
Let's view the coarse-grained beads we'll be simulating the scattering from.

In [ ]:
viewer = py3Dmol.view(width=600, height=400)
viewer.addModel(open(pdb_file).read(), "pdb")
viewer.setStyle({"cartoon": {"color": "spectrum"}, "stick": {"radius": 0.1}})
viewer.zoomTo()
viewer.show()

## 3. Interactive Simulation
Adjust the sliders below to see how the Kratky plot and Standard I(q) plot change!

In [ ]:
def update_saxs(solvent_density=0.334, hydration_shell_density=0.367, q_max=0.5):
    # Calculate profile
    q, intensity = calculate_saxs_profile(
        structure,
        q_min=0.01,
        q_max=q_max,
        n_points=100,
        include_solvent=True,
        solvent_density=solvent_density,
        hydration_shell_density=hydration_shell_density,
    )

    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Kratky Plot
    kratky = (q**2) * intensity
    axes[0].plot(q, kratky, "r-", lw=2)
    axes[0].set_title(r"Kratky Plot ($q^2 \cdot I(q)$ vs $q$)")
    axes[0].set_xlabel(r"$q$ ($\AA^{-1}$)")
    axes[0].set_ylabel(r"$q^2 \cdot I(q)$")
    axes[0].grid(True, alpha=0.3)

    # Standard I(q) Log Plot
    axes[1].semilogy(q, intensity, "b-", lw=2)
    axes[1].set_title(r"Standard SAXS Profile ($\log I(q)$ vs $q$)")
    axes[1].set_xlabel(r"$q$ ($\AA^{-1}$)")
    axes[1].set_ylabel(r"$\log I(q)$")
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Create interactive sliders
interact(
    update_saxs,
    solvent_density=FloatSlider(
        value=0.334,
        min=0.0,
        max=0.5,
        step=0.01,
        description="Solvent (e/Å³):",
        style={"description_width": "initial"},
    ),
    hydration_shell_density=FloatSlider(
        value=0.367,
        min=0.334,
        max=0.45,
        step=0.01,
        description="Shell (e/Å³):",
        style={"description_width": "initial"},
    ),
    q_max=FloatSlider(value=0.5, min=0.2, max=0.8, step=0.05, description="Q max (Å⁻¹):"),
);